In [ ]:
import pandas as pd
df0 = pd.read_csv('<DATA_ROOT>/MachineLearning/fail_ML_merge_20240221.csv')

In [ ]:
cols = ['FIPS', 'Crop', 'Irrigation Practice', 'year', 'fail_share',
       'Liability', 'Subsidy','RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3','RPL_THEME4','RPL_THEMES',
       'Thr0','spei180d-1', 'spei180d-2','spei270d-1', 'spei270d-2','dpi1', 'dpi5','dpi20','VPD-Mean',
       'VPD-1', 'VPD-2','Thr27', 'Thr30', 'Thr31','Thr33', 'Thr36']
dff = df0[cols]

dff = dff.dropna()
dff = dff[dff.fail_share > 0]
selected_crop = 'CORN' # 'WHEAT' , 'CORN' , 'SOYBEANS'
dff = dff[dff.Crop == selected_crop]

In [ ]:
import numpy as np
bins = list(dff.fail_share.quantile(np.arange(0,1,0.34)))
bins = bins + [1]
labels = list(np.arange(0,3))
dff['fail_cls'] = dff.fail_share.to_frame().apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
dff['fail_cls'] = dff['fail_cls'].fillna(0)


In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = dff.drop(columns=['FIPS','Crop','Irrigation Practice','year','fail_share','fail_cls'])
scaler = MinMaxScaler()
scaler.fit(df)
df_scaled = pd.DataFrame(scaler.transform(df), columns=df.columns)


In [ ]:
random_state_SOYBEANS = 85
params_SOYBEANS = {'max_depth': 7,
                  'learning_rate': 0.055,
                  'subsample': 0.65,
                  'colsample_bytree': 0.75,
                  'colsample_bylevel': 0.75,
                  'n_estimators': 650,
                  'max_leaves':100,
                  }

random_state_CORN = 85
params_CORN = {'max_depth': 7,
              'learning_rate': 0.05,
              'subsample': 0.8,
              'colsample_bytree': 0.75,
              'colsample_bylevel': 0.75,
              'n_estimators': 650,
              'max_leaves':100
               }

random_state_WHEAT = 100
params_WHEAT = {'max_depth': 7,
                'learning_rate': 0.007,
                'subsample': 0.8,
                'colsample_bytree': 0.55,
                'colsample_bylevel': 0.7,
                'n_estimators': 650,
                'max_leaves':100,
              }


In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df_scaled
y = dff['fail_cls']

if selected_crop == 'SOYBEANS':
  random_state = random_state_SOYBEANS
  params = params_SOYBEANS
elif selected_crop == 'CORN':
  random_state = random_state_CORN
  params = params_CORN
elif selected_crop == 'WHEAT':
  random_state = random_state_WHEAT
  params = params_WHEAT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1,random_state=random_state)

xgb_model = xgb.XGBClassifier(**params)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_pred2 = xgb_model.predict(X_train)

accuracy_test = accuracy_score(y_test, y_pred)
print("Accuracy_test: %.2f%%" % (accuracy_test * 100.0))
accuracy_train = accuracy_score(y_train, y_pred2)
print("Accuracy_train: %.2f%%" % (accuracy_train * 100.0))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(np.array([cm[0,0],cm[1,1],cm[2,2]])/np.sum(cm,axis=1))